# 04 · VIF, regresión polinómica y regularización sobre Ames Housing

**Módulo 3 · Sesión 7** — Regresión avanzada

## Objetivos

El notebook 03 mostró, sobre datos sintéticos, que Ridge y Lasso estabilizan coeficientes
inestables por colinealidad. Aquí se repite el ejercicio sobre datos reales: se diagnostica
multicolinealidad con **VIF** en Ames Housing, se agregan términos **polinómicos** que la
agravan a propósito, y se mide —no se asume— si Ridge, Lasso y Elastic Net (vía
`scikit-learn`, ya no a mano) mejoran la predicción, la estabilizan, o ninguna de las dos.

**Paquetes:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`, `statsmodels`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

SEMILLA = 42

## 1. Mismas variables del notebook 02, mismo split

Para poder comparar directamente contra la S6, se reutilizan las 11 variables numéricas y 3
categóricas de `02-regresion-multiple-aplicado.ipynb`, con el mismo `random_state`.

In [ ]:
datos = pd.read_csv("../datos/ames-housing.csv")

numericas = [
    "gr_liv_area", "total_bsmt_sf", "garage_area", "garage_cars",
    "overall_qual", "overall_cond", "year_built", "year_remod_add",
    "lot_area", "full_bath", "bedroom_abvgr",
]
categoricas = ["bldg_type", "house_style", "central_air"]

X = datos[numericas + categoricas]
y = datos["saleprice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)


def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

## 2. VIF de las variables originales

El VIF se calcula **solo sobre el entrenamiento** —es un diagnóstico de los predictores, y
calcularlo con datos de prueba sería asomarse a información que el modelo no debería usar
para tomar decisiones, el mismo principio del módulo 2.

In [ ]:
def calcular_vif(df_numerico):
    df_c = (df_numerico - df_numerico.mean()) / df_numerico.std()
    df_c.insert(0, "const", 1.0)
    vif = pd.DataFrame(
        {
            "variable": df_c.columns,
            "VIF": [
                variance_inflation_factor(df_c.values, i) for i in range(df_c.shape[1])
            ],
        }
    )
    return vif[vif["variable"] != "const"].sort_values("VIF", ascending=False)


vif_base = calcular_vif(X_train[numericas].dropna())
vif_base.round(2)

`garage_cars` y `garage_area` son las únicas por encima de 5 — exactamente el par que
`02-regresion-multiple-aplicado.ipynb` señaló como "miden lo mismo". El resto está por
debajo de 3.2: colinealidad moderada, ninguna severa (> 10). Con este nivel, y con $n$
grande frente a $p$, `03-multicolinealidad-polinomica.md` anticipa que la predicción no
debería resentirse mucho — se comprueba en la sección 4.

## 3. Términos polinómicos, y lo que le hacen al VIF

`overall_qual` y `gr_liv_area` son buenas candidatas a relación no lineal con el precio (una
casa de calidad 9 no vale simplemente 9/5 de una de calidad 5). Se agregan sus cuadrados.

In [ ]:
datos_poli = datos.copy()
datos_poli["gr_liv_area_2"] = datos_poli["gr_liv_area"] ** 2
datos_poli["overall_qual_2"] = datos_poli["overall_qual"] ** 2
numericas_poli = numericas + ["gr_liv_area_2", "overall_qual_2"]

X_poli = datos_poli[numericas_poli + categoricas]
Xp_train, Xp_test, y_train, y_test = train_test_split(
    X_poli, y, test_size=0.2, random_state=SEMILLA
)

vif_poli = calcular_vif(Xp_train[numericas_poli].dropna())
vif_poli.round(2)

`overall_qual` pasa de VIF ≈ 2.7 a **≈ 49**, y `gr_liv_area` de ≈ 3.2 a **≈ 20**. Elevar al
cuadrado no agrega información independiente: por construcción, $x$ y $x^2$ están casi
linealmente relacionadas en el rango donde vive la mayoría de los datos.

## 4. ¿Ayudan los términos polinómicos a predecir, a pesar del VIF?

In [ ]:
transformador_poli = ColumnTransformer(
    [
        (
            "num",
            Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]),
            numericas_poli,
        ),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categoricas),
    ]
)

modelo_ols_poli = Pipeline([("prep", transformador_poli), ("reg", LinearRegression())])
modelo_ols_poli.fit(Xp_train, y_train)
rmse_ols_poli = rmse(y_test, modelo_ols_poli.predict(Xp_test))

print(f"RMSE sin polinómicos (notebook 02):  $37,940")
print(f"RMSE con polinómicos (OLS):          ${rmse_ols_poli:,.0f}")

Con las mismas 14 variables más dos cuadrados, el RMSE baja de \$37,940 a bastante menos:
sí hay curvatura real que el modelo lineal en $x$ no capturaba. El costo es lo que se ve a
continuación.

In [ ]:
nombres_feat = modelo_ols_poli.named_steps["prep"].get_feature_names_out()
coef_ols = modelo_ols_poli.named_steps["reg"].coef_
i_qual = list(nombres_feat).index("num__overall_qual")
i_qual2 = list(nombres_feat).index("num__overall_qual_2")

print(f"beta(overall_qual)   = {coef_ols[i_qual]:>12,.0f}")
print(f"beta(overall_qual^2) = {coef_ols[i_qual2]:>12,.0f}")

Dos coeficientes enormes, de signo opuesto, para variables que miden casi lo mismo — la
firma exacta de la inestabilidad por colinealidad que predice `03-multicolinealidad-polinomica.md`.
El modelo predice bien (la suma de sus efectos es razonable), pero ningún coeficiente por
separado se puede interpretar como "el efecto de la calidad".

## 5. ¿Lo arregla la regularización?

Se aparta una porción **del entrenamiento** —nunca del test— para explorar el efecto de
$\lambda$. Es un solo split de validación, deliberadamente simple; la sesión 8 muestra por
qué un solo split es ruidoso y lo reemplaza por validación cruzada.

In [ ]:
Xp_sub, Xp_val, y_sub, y_val = train_test_split(
    Xp_train, y_train, test_size=0.25, random_state=SEMILLA
)

lambdas = np.logspace(-2, 3, 20)
resultados = []
for lam in lambdas:
    for nombre, Modelo in [("Ridge", Ridge), ("Lasso", Lasso)]:
        kwargs = {"max_iter": 20000} if nombre == "Lasso" else {}
        m = Pipeline([("prep", transformador_poli), ("reg", Modelo(alpha=lam, **kwargs))])
        m.fit(Xp_sub, y_sub)
        c = m.named_steps["reg"].coef_
        resultados.append(
            {
                "lambda": lam,
                "modelo": nombre,
                "rmse_val": rmse(y_val, m.predict(Xp_val)),
                "overall_qual": c[i_qual],
                "overall_qual_2": c[i_qual2],
            }
        )
resultados = pd.DataFrame(resultados)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))

for nombre, grupo in resultados.groupby("modelo"):
    ejes[0].plot(grupo["lambda"], grupo["rmse_val"], marker="o", markersize=3, label=nombre)
ejes[0].set_xscale("log")
ejes[0].set_xlabel("$\\lambda$")
ejes[0].set_ylabel("RMSE de validación ($)")
ejes[0].set_title("¿Mejora la predicción?")
ejes[0].legend()

ridge_res = resultados[resultados["modelo"] == "Ridge"]
lasso_res = resultados[resultados["modelo"] == "Lasso"]
ejes[1].plot(ridge_res["lambda"], ridge_res["overall_qual"], "o-", ms=3, label="Ridge: qual")
ejes[1].plot(ridge_res["lambda"], ridge_res["overall_qual_2"], "o-", ms=3, label="Ridge: qual²")
ejes[1].plot(lasso_res["lambda"], lasso_res["overall_qual"], "s--", ms=3, label="Lasso: qual")
ejes[1].plot(lasso_res["lambda"], lasso_res["overall_qual_2"], "s--", ms=3, label="Lasso: qual²")
ejes[1].axhline(0, color="black", linewidth=0.8)
ejes[1].set_xscale("log")
ejes[1].set_xlabel("$\\lambda$")
ejes[1].set_ylabel("Coeficiente")
ejes[1].set_title("¿Se estabiliza el par colineal?")
ejes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

Dos lecturas distintas de la misma trayectoria:

- **Predicción (panel izquierdo):** el RMSE de validación no mejora con $\lambda$ creciente
  — el mínimo está cerca de $\lambda \to 0$, es decir, cerca de OLS. Regularizar más fuerte
  solo empeora la predicción en este dataset. Es la confirmación empírica, con datos reales,
  de que la multicolinealidad aquí no está dañando la predicción (`03-multicolinealidad-polinomica.md`,
  sección 2) — porque $n$ es grande frente a $p$.
- **Coeficientes (panel derecho):** a pesar de eso, la trayectoria sí cambia mucho el par
  `overall_qual`/`overall_qual²`. Ridge reacciona con relativamente poco $\lambda$: hacia
  $\lambda \approx 100$ el signo ya se corrigió (`overall_qual` pasa a positivo), y hacia
  $\lambda=1000$ ambos quedan de magnitud similar (≈11 mil y ≈14 mil). Lasso necesita mucho
  más $\lambda$ para reaccionar —su penalización crece más despacio que la de Ridge cerca de
  cero— y termina **anulando** `overall_qual` por completo hacia $\lambda=1000$, dejando que
  `overall_qual²` cargue sola con el efecto de la calidad.

La lección no es "la regularización siempre ayuda a predecir". Es: regulariza la
**interpretación** de los coeficientes con un costo bajo (o nulo) en la predicción, cuando
—como aquí— hay datos suficientes para que OLS ya generalizara razonablemente bien.

## 6. Comparación final en el conjunto de prueba

Se elige $\lambda=50$ —un punto donde el par colineal ya está visiblemente más estable, sin
llegar al extremo— y se entrena con **todo** `X_train`, evaluando en el `X_test` que ningún
paso anterior tocó.

In [ ]:
LAMBDA_ELEGIDO = 50
modelos_finales = {
    "OLS": LinearRegression(),
    "Ridge (λ=50)": Ridge(alpha=LAMBDA_ELEGIDO),
    "Lasso (λ=50)": Lasso(alpha=LAMBDA_ELEGIDO, max_iter=20000),
}

comparacion_final = []
for nombre, reg in modelos_finales.items():
    m = Pipeline([("prep", transformador_poli), ("reg", reg)])
    m.fit(Xp_train, y_train)
    pred = m.predict(Xp_test)
    comparacion_final.append(
        {"modelo": nombre, "RMSE": rmse(y_test, pred), "MAE": mean_absolute_error(y_test, pred)}
    )

pd.DataFrame(comparacion_final).round(0)

Lasso queda prácticamente empatado con OLS; Ridge, algo peor. Ninguno mejora de forma
contundente al OLS con polinómicos — coherente con lo que ya mostró la curva de validación.
El único punto de fricción: **la validación sugería $\lambda$ cercano a 0, no 50** — que
aquí, sobre este test set puntual, Lasso a $\lambda=50$ termine tan cerca de OLS es en parte
suerte del split. Un solo conjunto de prueba es una muestra; su RMSE tiene su propia
variabilidad, del mismo tipo que hizo inestables los coeficientes de OLS en el notebook 03.
Repetir la validación sobre varias particiones —en vez de una sola— es exactamente lo que
hace la validación cruzada de la sesión 8.

**Nota sobre Elastic Net.** `scikit-learn` combina ambas penalizaciones con una escala de
`alpha` distinta a Ridge o Lasso por separado —no es directamente comparable al mismo
$\lambda=50$—; requiere su propio barrido. Queda para el ejercicio 02 del módulo.

## Resumen

| Resultado | Conecta con |
|---|---|
| `garage_cars`/`garage_area`: único par con VIF > 5 en las variables originales | Confirma el hallazgo cualitativo del notebook 02 |
| Agregar $x^2$ dispara el VIF de esa variable a ≈ 50, pero baja el RMSE de \$37,940 a mucho menos | Regresión polinómica sigue siendo lineal en $\boldsymbol{\beta}$ |
| RMSE de validación no mejora con $\lambda$: la multicolinealidad aquí no daña la predicción | `03-multicolinealidad-polinomica.md`, sección 2 |
| Ridge y Lasso sí estabilizan el par colineal, a ritmos muy distintos | `04-regularizacion.md`, geometría $L_2$ vs. $L_1$ |
| Un solo split de validación/prueba es ruidoso | Motivación directa de la validación cruzada (sesión 8) |